# 🪨 GeoProspecting — Análise de Favorabilidade Exploratória Mineral
## Estudo de Caso: Prospecção de Ouro (Fase Inicial)

---

**Metodologia baseada em:** PSI Analytics — Análise Integrada de Favorabilidade Exploratória  
**Dados:** Sintéticos (demonstração do pipeline; não representam área real)  
**Commodity alvo:** Ouro (Au)

### Objetivo
Demonstrar o pipeline completo de análise de favorabilidade mineral na fase inicial de exploração, integrando dados geofísicos e radiométricos para identificar **zonas prioritárias** e gerar um **ranking de alvos**.

### Etapas do Estudo de Caso
1. Instalação e importação de bibliotecas
2. Geração de dados sintéticos (ou carga de dados reais)
3. Normalização robusta dos dados
4. Cálculo do PSIIndex (índice de favorabilidade para ouro)
5. Ternário radiométrico K-U-Th
6. Identificação de zonas prioritárias (clusters top 5%)
7. Ranking de subalvos recomendados
8. Geração do relatório técnico (mapas + tabelas)
9. Exportação dos resultados (CSV + PDF)

---
> ⚠️ **IMPORTANTE:** Este notebook usa dados **SINTÉTICOS** para fins didáticos. Em uso real, substitua pela seção de carregamento de dados reais (GeoTIFF ou CSV).

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalar dependências (necessário apenas no Colab)
!pip install numpy scipy matplotlib pandas scikit-learn fpdf2 rasterio --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter, label, maximum_filter
from scipy.spatial.distance import cdist
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 8

print('✅ Bibliotecas importadas com sucesso!')

## 2. Configuração da Área de Estudo e Alvos

Defina aqui as coordenadas da área de interesse (bounding box) e os pontos-alvo a serem analisados.

In [ ]:
# ============================================================
# CONFIGURAÇÕES — EDITE CONFORME SUA ÁREA DE INTERESSE
# ============================================================

# Bounding box (lon_min, lat_min, lon_max, lat_max)
BBOX = (-41.95, -4.75, -40.30, -3.90)

# Resolução da grade (graus por pixel)
RESOLUCAO = 0.02  # ~2 km por pixel

# Pontos de interesse (alvos a analisar)
ALVOS = {
    'T1': (-40.57, -4.65),   # (lon, lat)
    'T2': (-41.58, -4.30),
    'T3': (-41.20, -4.52),
}

# Raio de análise por alvo (km)
RAIO_KM = 20

# Commodity alvo
COMMODITY = 'OURO'

# ============================================================
# Geração da grade de coordenadas
# ============================================================
lon_min, lat_min, lon_max, lat_max = BBOX

lons = np.arange(lon_min, lon_max, RESOLUCAO)
lats = np.arange(lat_min, lat_max, RESOLUCAO)
LON, LAT = np.meshgrid(lons, lats)

NX, NY = LON.shape
print(f'✅ Grade criada: {NX} x {NY} pixels')
print(f'   Área: {lon_min:.2f}°W a {lon_max:.2f}°W | {lat_min:.2f}°S a {lat_max:.2f}°S')
print(f'   Resolução: ~{RESOLUCAO * 111:.1f} km/pixel')
print(f'   Alvos: {list(ALVOS.keys())}')

## 3. Dados de Entrada

### 3A. Dados Sintéticos (demonstração)
Geração de dados sintéticos com assinaturas geofísicas típicas de sistemas auríferos:
- **K elevado** nas zonas de alteração hidrotermal
- **Gradientes magnéticos** nas bordas de corpos intrusivos
- **Desacoplamento K-MAG/GRAV** como indicador de alteração

### 3B. Como usar dados reais (GeoTIFF/CSV)
Veja a célula de carregamento real abaixo — basta substituir os arrays sintéticos.

In [ ]:
np.random.seed(42)

def distancia_km(lon1, lat1, lon2, lat2):
    """Distância aproximada em km entre dois pontos (projeção plana)."""
    dlat = (lat2 - lat1) * 111.0
    dlon = (lon2 - lon1) * 111.0 * np.cos(np.radians((lat1 + lat2) / 2))
    return np.sqrt(dlat**2 + dlon**2)

def gaussiana_2d(lon_c, lat_c, sigma_km=15, amplitude=1.0):
    """Cria uma anomalia gaussiana centrada em (lon_c, lat_c)."""
    sigma_deg = sigma_km / 111.0
    return amplitude * np.exp(-((LON - lon_c)**2 + (LAT - lat_c)**2) / (2 * sigma_deg**2))

# --- Magnetometria (MAG): anomalias magnéticas regionais ---
MAG = (gaussiana_2d(-40.6, -4.65, sigma_km=20, amplitude=0.8)
     + gaussiana_2d(-41.55, -4.28, sigma_km=18, amplitude=0.9)
     + gaussiana_2d(-41.18, -4.50, sigma_km=15, amplitude=0.7)
     + 0.15 * np.random.randn(*LON.shape))

# --- Gravimetria (GRAV): resposta gravitacional ---
GRAV = (gaussiana_2d(-40.58, -4.68, sigma_km=25, amplitude=0.7)
      + gaussiana_2d(-41.60, -4.35, sigma_km=22, amplitude=0.85)
      + gaussiana_2d(-41.22, -4.55, sigma_km=18, amplitude=0.65)
      + 0.12 * np.random.randn(*LON.shape))

# --- Anomalia Bouguer ---
BOUGUER = GRAV * 0.9 + 0.1 * np.random.randn(*LON.shape)

# --- Radiometria K (Potássio): indicador de alteração hidrotermal ---
# Para ouro: K ALTO é o sinal mais importante (alteração potássica)
K = (gaussiana_2d(-40.57, -4.64, sigma_km=12, amplitude=1.0)   # T1 — K alto
   + gaussiana_2d(-40.54, -4.70, sigma_km=8,  amplitude=0.85)
   + gaussiana_2d(-41.58, -4.18, sigma_km=10, amplitude=0.95)  # T2 — K alto
   + gaussiana_2d(-41.47, -4.33, sigma_km=9,  amplitude=0.80)
   + gaussiana_2d(-41.24, -4.52, sigma_km=11, amplitude=0.90)  # T3 — K alto
   + gaussiana_2d(-41.13, -4.55, sigma_km=7,  amplitude=0.75)
   + 0.10 * np.random.randn(*LON.shape))

# --- Radiometria U (Urânio) ---
U = (gaussiana_2d(-40.55, -4.62, sigma_km=14, amplitude=0.6)
   + gaussiana_2d(-41.50, -4.25, sigma_km=12, amplitude=0.55)
   + gaussiana_2d(-41.20, -4.50, sigma_km=10, amplitude=0.65)
   + 0.12 * np.random.randn(*LON.shape))

# --- Radiometria Th (Tório) ---
Th = (gaussiana_2d(-40.60, -4.67, sigma_km=16, amplitude=0.5)
    + gaussiana_2d(-41.55, -4.30, sigma_km=14, amplitude=0.6)
    + gaussiana_2d(-41.22, -4.53, sigma_km=12, amplitude=0.55)
    + 0.10 * np.random.randn(*LON.shape))

# Clip para evitar valores negativos
MAG = np.clip(MAG, 0, None)
GRAV = np.clip(GRAV, 0, None)
BOUGUER = np.clip(BOUGUER, 0, None)
K = np.clip(K, 0, None)
U = np.clip(U, 0, None)
Th = np.clip(Th, 0, None)

dados_brutos = {'MAG': MAG, 'GRAV': GRAV, 'BOUGUER': BOUGUER, 'K': K, 'U': U, 'Th': Th}

print('✅ Dados sintéticos gerados!')
print('   Camadas:', list(dados_brutos.keys()))
print('   Forma das grades:', MAG.shape)

In [ ]:
# ============================================================
# 3B. CARREGAMENTO DE DADOS REAIS (descomente para usar)
# ============================================================
# Para usar dados reais, faça upload dos arquivos GeoTIFF ou CSV
# e descomente o bloco abaixo.
#
# --- Opção 1: GeoTIFF (via rasterio) ---
# import rasterio
# from google.colab import files
#
# uploaded = files.upload()  # Faz upload dos arquivos
#
# def ler_geotiff(path):
#     with rasterio.open(path) as src:
#         data = src.read(1).astype(float)
#         data[data == src.nodata] = np.nan
#     return data
#
# MAG     = ler_geotiff('magnetometria.tif')
# GRAV    = ler_geotiff('gravimetria.tif')
# BOUGUER = ler_geotiff('bouguer.tif')
# K       = ler_geotiff('radiometria_K.tif')
# U       = ler_geotiff('radiometria_U.tif')
# Th      = ler_geotiff('radiometria_Th.tif')
#
# --- Opção 2: CSV tabular ---
# df = pd.read_csv('dados_geofisicos.csv')
# # Colunas esperadas: lon, lat, mag, grav, bouguer, K, U, Th
# # (interpolação para grade regular se necessário)

print('ℹ️  Usando dados sintéticos. Para dados reais, descomente o bloco acima.')

## 4. Normalização Robusta dos Dados

A normalização robusta usa a **mediana e o IQR** (intervalo interquartil) em vez de média e desvio padrão, tornando-a resistente a outliers — comum em dados geofísicos com anomalias extremas.

In [ ]:
def normalizar_robusto(arr):
    """Normalização robusta 0-1 baseada em percentis (p2 a p98)."""
    arr = arr.copy().astype(float)
    mascara_nan = np.isnan(arr)
    p2  = np.nanpercentile(arr, 2)
    p98 = np.nanpercentile(arr, 98)
    arr_norm = (arr - p2) / (p98 - p2 + 1e-10)
    arr_norm = np.clip(arr_norm, 0, 1)
    arr_norm[mascara_nan] = np.nan
    return arr_norm

# Normalizar todas as camadas
dados_norm = {k: normalizar_robusto(v) for k, v in dados_brutos.items()}

# Verificação
print('✅ Normalização robusta aplicada (0–1):')
for nome, arr in dados_norm.items():
    print(f'   {nome}: min={arr.min():.3f}  max={arr.max():.3f}  média={arr.mean():.3f}')

In [ ]:
# Plotar mapas por camada
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Mapas por Camada — Normalização Robusta 0–1  |  {COMMODITY} (DADOS SINTÉTICOS)',
             fontsize=12, fontweight='bold', y=1.01)

cmaps = {'MAG': 'RdBu_r', 'GRAV': 'viridis', 'BOUGUER': 'plasma',
         'K': 'YlOrRd', 'U': 'BuGn', 'Th': 'PuBu'}

for ax, (nome, arr) in zip(axes.flat, dados_norm.items()):
    im = ax.pcolormesh(lons, lats, arr, cmap=cmaps[nome], shading='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Valor normalizado')
    # Plotar alvos
    for alvo_nome, (alon, alat) in ALVOS.items():
        ax.plot(alon, alat, 'w^', ms=8, zorder=5)
        ax.text(alon, alat + 0.03, alvo_nome, color='white', fontsize=7,
                ha='center', fontweight='bold')
    ax.set_title(nome, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('mapa_camadas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Mapa salvo: mapa_camadas.png')

## 5. Ternário Radiométrico K-U-Th

O ternário RGB (R=K, G=U, B=Th) é uma ferramenta clássica de mapeamento geológico:
- **Vermelho intenso** → dominância de K (alteração potássica — favorável para Au)
- **Verde intenso** → dominância de U (associado a granitos evoluídos)
- **Azul intenso** → dominância de Th (rochas máficas/ultramáficas)
- **Magenta** → K + Th (rochas félsicas não alteradas)
- **Amarelo** → K + U (granitos alterados hidrotermalmente)

In [ ]:
# Composição RGB ternária
R = dados_norm['K']
G = dados_norm['U']
B = dados_norm['Th']

rgb = np.dstack([R, G, B])
rgb = np.clip(rgb, 0, 1)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(rgb, extent=[lon_min, lon_max, lat_min, lat_max],
          origin='lower', aspect='auto')

for alvo_nome, (alon, alat) in ALVOS.items():
    ax.plot(alon, alat, 'w^', ms=10, zorder=5)
    ax.text(alon, alat + 0.04, alvo_nome, color='white', fontsize=9,
            ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

# Legenda
legenda = [
    Patch(color=(1,0,0), label='K (alteração potássica)'),
    Patch(color=(0,1,0), label='U (granitos evoluídos)'),
    Patch(color=(0,0,1), label='Th (máficas/ultramáficas)'),
]
ax.legend(handles=legenda, loc='lower right', fontsize=8,
          fancybox=True, framealpha=0.8)

ax.set_title(f'Ternário Radiométrico K-U-Th (RGB)  |  {COMMODITY} (DADOS SINTÉTICOS)',
             fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('ternario_KUTh.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Ternário salvo: ternario_KUTh.png')

## 6. Cálculo do PSIIndex — Índice de Favorabilidade para Ouro

O **PSIIndex** integra as camadas geofísicas com pesos orientados pela **assinatura geológica do ouro**:

$$PSIIndex = w_K \cdot K_{norm} + w_{grad} \cdot Grad + w_{MAG} \cdot MAG_{norm} + w_{GRAV} \cdot GRAV_{norm} + B_{desacopl}$$

Onde:
- $K_{norm}$ → potássio normalizado (alteração hidrotermal)
- $Grad$ → gradiente espacial do índice (bordas estruturais)
- $MAG_{norm}$, $GRAV_{norm}$ → geofísica estrutural
- $B_{desacopl}$ → bônus por desacoplamento K alto / MAG+GRAV baixos

In [ ]:
# ============================================================
# Pesos do PSIIndex para OURO
# Ajuste conforme modelo geológico da área
# ============================================================
PESOS = {
    'K':       0.40,   # Alteração potássica — indicador primário de Au
    'gradiente': 0.25, # Bordas/gradientes — controle estrutural
    'MAG':     0.15,   # Magnetometria — controle litológico
    'GRAV':    0.10,   # Gravimetria — controle crustal
    'U':       0.05,   # Urânio — complementar
    'Th':      0.05,   # Tório — complementar
}

# --- Suavização dos dados (reduz ruído de alta frequência) ---
sigma_suav = 1.5  # pixels
K_suav   = gaussian_filter(dados_norm['K'],   sigma=sigma_suav)
MAG_suav = gaussian_filter(dados_norm['MAG'], sigma=sigma_suav)
GRAV_suav= gaussian_filter(dados_norm['GRAV'],sigma=sigma_suav)
U_suav   = gaussian_filter(dados_norm['U'],   sigma=sigma_suav)
Th_suav  = gaussian_filter(dados_norm['Th'],  sigma=sigma_suav)

# --- Gradiente espacial (bordas e transições) ---
gx, gy = np.gradient(K_suav)
GRAD = np.sqrt(gx**2 + gy**2)
GRAD = normalizar_robusto(GRAD)

# --- Índice base ---
PSI_base = (PESOS['K']        * K_suav
          + PESOS['gradiente'] * GRAD
          + PESOS['MAG']       * MAG_suav
          + PESOS['GRAV']      * GRAV_suav
          + PESOS['U']         * U_suav
          + PESOS['Th']        * Th_suav)

# --- Bônus de desacoplamento K alto / MAG+GRAV baixos ---
# Indica alteração hidrotermal sem resposta gravitacional/magnética forte
# Padrão típico de sistemas auríferos epitermais/mesozonais
MAG_GRAV_medio = (MAG_suav + GRAV_suav) / 2
BONUS_DESACOPL = np.where(
    (K_suav > 0.6) & (MAG_GRAV_medio < 0.5),
    0.08 * K_suav,   # Bônus de até 8%
    0.0
)

# --- PSIIndex final ---
PSIIndex = np.clip(PSI_base + BONUS_DESACOPL, 0, 1)
PSIIndex = normalizar_robusto(PSIIndex)

print(f'✅ PSIIndex calculado!')
print(f'   Min: {PSIIndex.min():.4f}  |  Max: {PSIIndex.max():.4f}  |  Média: {PSIIndex.mean():.4f}')
print(f'   Pesos aplicados: {PESOS}')

In [ ]:
# Plotar PSIIndex
fig, ax = plt.subplots(figsize=(10, 7))

im = ax.pcolormesh(lons, lats, PSIIndex, cmap='hot_r', shading='auto', vmin=0.1, vmax=1)
cbar = plt.colorbar(im, ax=ax, label='PSIIndex (favorabilidade relativa)', shrink=0.8)

# Contorno de alta favorabilidade
cs = ax.contour(lons, lats, PSIIndex, levels=[0.7, 0.8, 0.9],
                colors=['cyan', 'lime', 'white'], linewidths=[0.8, 1.0, 1.2])
ax.clabel(cs, fmt='%.1f', fontsize=7)

# Alvos
for alvo_nome, (alon, alat) in ALVOS.items():
    ax.plot(alon, alat, 'c^', ms=12, zorder=6, markeredgecolor='black', markeredgewidth=1)
    ax.text(alon, alat + 0.04, alvo_nome, color='cyan', fontsize=10,
            ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='black', alpha=0.6))

ax.set_title(f'PSIIndex ({COMMODITY}) — Índice de Favorabilidade Exploratória  |  DADOS SINTÉTICOS',
             fontweight='bold', fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('psiindex.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ PSIIndex salvo: psiindex.png')

## 7. Identificação de Zonas Prioritárias (Clusters Top 5%)

Para cada alvo, identificamos as **zonas contíguas de alta favorabilidade** dentro do raio de análise, considerando apenas os pixels no **top 5%** do PSIIndex.

In [ ]:
def graus_para_km(dlon, dlat, lat_ref):
    """Converte diferença em graus para km."""
    km_lat = dlat * 111.0
    km_lon = dlon * 111.0 * np.cos(np.radians(lat_ref))
    return np.sqrt(km_lat**2 + km_lon**2)

def encontrar_zonas(alvo_nome, alon, alat, raio_km, psi, lons_arr, lats_arr, top_pct=0.05, max_zonas=5):
    """Encontra zonas prioritárias dentro do raio para um alvo."""
    LON_g, LAT_g = np.meshgrid(lons_arr, lats_arr)

    # Máscara de raio
    dist = graus_para_km(LON_g - alon, LAT_g - alat, alat)
    mascara_raio = dist <= raio_km

    # Pixels dentro do raio
    psi_raio = np.where(mascara_raio, psi, np.nan)

    # Limiar top N%
    limiar = np.nanpercentile(psi_raio, (1 - top_pct) * 100)

    # Binarizar: top N% dentro do raio
    mapa_binario = (psi_raio >= limiar).astype(int)

    # Rotular regiões conectadas (zonas)
    mapa_labels, n_zonas = label(mapa_binario)

    zonas = []
    for z_id in range(1, n_zonas + 1):
        mascara_zona = (mapa_labels == z_id)
        pixels_psi = psi[mascara_zona]
        if len(pixels_psi) == 0:
            continue

        lons_zona = LON_g[mascara_zona]
        lats_zona = LAT_g[mascara_zona]
        area_km2 = len(pixels_psi) * (RESOLUCAO * 111)**2

        # PriorityScore: combina pico, média, área (em menor peso)
        peak = pixels_psi.max()
        mean = pixels_psi.mean()
        area_norm = min(area_km2 / 10.0, 1.0)  # normaliza área até 10 km²
        priority = 0.50 * peak + 0.35 * mean + 0.15 * area_norm

        zonas.append({
            'Target': alvo_nome,
            'Zone': z_id,
            'PriorityScore': round(priority, 4),
            'PeakScore':     round(peak, 4),
            'MeanScore':     round(mean, 4),
            'Area_km2':      round(area_km2, 4),
            'CentroidLon':   round(lons_zona.mean(), 4),
            'CentroidLat':   round(lats_zona.mean(), 4),
            '_mascara':      mascara_zona,
        })

    # Ordenar por PriorityScore, retornar top N
    zonas.sort(key=lambda x: x['PriorityScore'], reverse=True)
    return zonas[:max_zonas]


# Encontrar zonas para cada alvo
todas_zonas = []
for alvo_nome, (alon, alat) in ALVOS.items():
    zonas = encontrar_zonas(alvo_nome, alon, alat, RAIO_KM, PSIIndex, lons, lats)
    todas_zonas.extend(zonas)
    print(f'  {alvo_nome}: {len(zonas)} zonas encontradas')

# Criar DataFrame (sem a coluna de máscara)
df_zonas = pd.DataFrame([{k: v for k, v in z.items() if k != '_mascara'} for z in todas_zonas])
df_zonas = df_zonas.sort_values('PriorityScore', ascending=False).reset_index(drop=True)

print(f'\n✅ Total de zonas identificadas: {len(df_zonas)}')
print(df_zonas.to_string(index=False))

## 8. Ranking de Subalvos Recomendados

Dentro de cada zona, identificamos os **máximos locais** espaçados por no mínimo 5 km, que representam os **pontos ótimos de investigação de campo**.

In [ ]:
def encontrar_subalvos(alvo_nome, alon, alat, raio_km, psi, lons_arr, lats_arr,
                       espac_min_km=5.0, n_max=6):
    """Encontra máximos locais espaçados dentro do raio."""
    LON_g, LAT_g = np.meshgrid(lons_arr, lats_arr)

    dist = graus_para_km(LON_g - alon, LAT_g - alat, alat)
    mascara_raio = dist <= raio_km

    psi_raio = np.where(mascara_raio, psi, 0)

    # Detectar máximos locais (vizinhança 5x5 pixels)
    psi_max = maximum_filter(psi_raio, size=5)
    locais_max = (psi_raio == psi_max) & (psi_raio > 0.6) & mascara_raio

    # Coordenadas dos máximos
    ys, xs = np.where(locais_max)
    if len(ys) == 0:
        return []

    pontos = []
    for y, x in zip(ys, xs):
        pontos.append({
            'lon': lons_arr[x],
            'lat': lats_arr[y],
            'score': psi_raio[y, x]
        })

    # Ordenar por score
    pontos.sort(key=lambda p: p['score'], reverse=True)

    # Filtrar por espaçamento mínimo
    selecionados = []
    for p in pontos:
        if len(selecionados) >= n_max:
            break
        aceito = True
        for s in selecionados:
            d = graus_para_km(p['lon'] - s['lon'], p['lat'] - s['lat'],
                              (p['lat'] + s['lat']) / 2)
            if d < espac_min_km:
                aceito = False
                break
        if aceito:
            selecionados.append(p)

    return [{
        'Target': alvo_nome,
        'Rank': i + 1,
        'Score': round(p['score'], 5),
        'Lon': round(p['lon'], 5),
        'Lat': round(p['lat'], 5),
    } for i, p in enumerate(selecionados)]


# Encontrar subalvos
todos_subalvos = []
for alvo_nome, (alon, alat) in ALVOS.items():
    subs = encontrar_subalvos(alvo_nome, alon, alat, RAIO_KM, PSIIndex, lons, lats)
    todos_subalvos.extend(subs)
    print(f'  {alvo_nome}: {len(subs)} subalvos recomendados')

df_subalvos = pd.DataFrame(todos_subalvos)

print(f'\n✅ Total de subalvos: {len(df_subalvos)}')
print(df_subalvos.to_string(index=False))

## 9. Tabela Comparativa dos Alvos

Resumo estatístico do PSIIndex para cada alvo dentro do raio de 5 km.

In [ ]:
RAIO_TABELA_KM = 5  # raio para a tabela comparativa

LON_g, LAT_g = np.meshgrid(lons, lats)

registros = []
for alvo_nome, (alon, alat) in ALVOS.items():
    dist = graus_para_km(LON_g - alon, LAT_g - alat, alat)
    mascara = dist <= RAIO_TABELA_KM
    psi_local = PSIIndex[mascara]

    registros.append({
        'Target':      alvo_nome,
        'LocalMean':   round(psi_local.mean(), 3),
        'P90':         round(np.percentile(psi_local, 90), 3),
        'Max':         round(psi_local.max(), 3),
        'Consistency': round(psi_local.mean() / (psi_local.std() + 1e-6), 3),
        'Dominance':   round(psi_local.max() / (1 - psi_local.mean() + 1e-6), 3),
        'Risk':        round((1 - psi_local.mean()) / (psi_local.std() + 1e-6), 3),
    })

df_alvos = pd.DataFrame(registros).sort_values('LocalMean', ascending=False).reset_index(drop=True)
print(f'Tabela comparativa dos alvos (raio {RAIO_TABELA_KM} km):')
print(df_alvos.to_string(index=False))

In [ ]:
# Visualizar tabela como gráfico de barras
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cores = ['#e74c3c', '#3498db', '#2ecc71']

metricas = ['LocalMean', 'P90', 'Max']
titulos  = ['Média Local (5 km)', 'Percentil 90', 'Score Máximo']

for ax, met, tit in zip(axes, metricas, titulos):
    bars = ax.bar(df_alvos['Target'], df_alvos[met], color=cores, edgecolor='black', linewidth=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(tit, fontweight='bold')
    ax.set_ylabel('PSIIndex')
    ax.axhline(0.7, color='orange', linestyle='--', linewidth=1, label='Limiar 0.7')
    ax.axhline(0.8, color='red',    linestyle='--', linewidth=1, label='Limiar 0.8')
    for bar, val in zip(bars, df_alvos[met]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

axes[0].legend(fontsize=7)
fig.suptitle(f'Comparação de Alvos — PSIIndex ({COMMODITY})  |  DADOS SINTÉTICOS',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('comparacao_alvos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: comparacao_alvos.png')

## 10. Painéis Dedicados por Alvo

Para cada alvo: PSIIndex (zoom), K (alteração), Gradiente e Zonas top 5%.

In [ ]:
def painel_alvo(alvo_nome, alon, alat, raio_km=6):
    """Gera painel dedicado de 4 subplots para um alvo."""
    # Extensão do zoom
    delta = raio_km / 111.0
    ext = [alon - delta, alon + delta, alat - delta, alat + delta]

    # Máscara de pixels no zoom
    LON_g, LAT_g = np.meshgrid(lons, lats)
    mask_zoom = ((LON_g >= ext[0]) & (LON_g <= ext[1]) &
                 (LAT_g >= ext[2]) & (LAT_g <= ext[3]))

    # Zonas top 5% dentro do raio
    dist = graus_para_km(LON_g - alon, LAT_g - alat, alat)
    psi_raio = np.where(dist <= RAIO_KM, PSIIndex, np.nan)
    limiar = np.nanpercentile(psi_raio, 95)
    mapa_zonas = PSIIndex >= limiar

    lons_zoom = lons[(lons >= ext[0]) & (lons <= ext[1])]
    lats_zoom = lats[(lats >= ext[2]) & (lats <= ext[3])]

    def crop(arr):
        return arr[np.ix_(
            (lats >= ext[2]) & (lats <= ext[3]),
            (lons >= ext[0]) & (lons <= ext[1])
        )]

    PSI_z  = crop(PSIIndex)
    K_z    = crop(dados_norm['K'])
    GRAD_z = crop(GRAD)
    ZON_z  = crop(mapa_zonas.astype(float))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f'Painel Dedicado {alvo_nome}  |  {COMMODITY} (DADOS SINTÉTICOS)',
                 fontweight='bold', fontsize=11)

    configs = [
        (PSI_z,  'hot_r',   f'{alvo_nome}  PSIIndex (zoom ±~{raio_km} km)'),
        (K_z,    'YlOrRd',  f'{alvo_nome}  K (alteração) (zoom)'),
        (GRAD_z, 'plasma',  f'{alvo_nome}  Gradiente do índice (zoom)'),
        (ZON_z,  'Greens',  f'{alvo_nome}  Zonas (top 5% dentro {RAIO_KM} km)'),
    ]

    for ax, (arr, cmap, titulo) in zip(axes, configs):
        im = ax.pcolormesh(lons_zoom, lats_zoom, arr, cmap=cmap, shading='auto')
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.plot(alon, alat, 'c^', ms=10, zorder=5, markeredgecolor='black')
        ax.set_title(titulo, fontsize=8)
        ax.set_xlabel('Lon')
        ax.set_ylabel('Lat')

    plt.tight_layout()
    fname = f'painel_{alvo_nome}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Painel salvo: {fname}')


for alvo_nome, (alon, alat) in ALVOS.items():
    painel_alvo(alvo_nome, alon, alat)

## 11. Exportação dos Resultados

In [ ]:
# Exportar CSVs
df_zonas.to_csv('zonas_prioritarias.csv', index=False)
df_subalvos.to_csv('subalvos_recomendados.csv', index=False)
df_alvos.to_csv('tabela_alvos.csv', index=False)

print('✅ CSVs exportados:')
print('   zonas_prioritarias.csv')
print('   subalvos_recomendados.csv')
print('   tabela_alvos.csv')

In [ ]:
# Gerar relatório técnico em PDF
from fpdf import FPDF
import os
from datetime import date

class RelatorioPDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'GeoProspecting — Análise de Favorabilidade Exploratória Mineral', ln=True, align='C')
        self.set_font('Helvetica', '', 8)
        self.cell(0, 6, f'Commodity: {COMMODITY}  |  Dados: SINTÉTICOS (demonstração)  |  Data: {date.today()}',
                  ln=True, align='C')
        self.ln(3)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-12)
        self.set_font('Helvetica', 'I', 7)
        self.cell(0, 6, f'Página {self.page_no()} — DADOS SINTÉTICOS — Não representa área real', align='C')


pdf = RelatorioPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# --- Página 1: Capa e escopo ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, f'Relatório GeoProspecting — {COMMODITY}', ln=True, align='C')
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 8, f'Área (BBox lon/lat): ({lon_min}, {lat_min}, {lon_max}, {lat_max})', ln=True, align='C')
pdf.ln(5)

pdf.set_font('Helvetica', 'B', 10)
pdf.cell(0, 8, 'IMPORTANTE:', ln=True)
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 6,
    'Dados SINTÉTICOS gerados para demonstração do pipeline; não representam uma área real.\n'
    f'Assinatura (Au): ênfase em K (alteração), gradientes/bordas e desacoplamento\n'
    '(K alto sem MAG/GRAV altos), visando refletir padrões típicos de sistemas auríferos.')
pdf.ln(4)

pdf.set_font('Helvetica', 'B', 10)
pdf.cell(0, 7, 'Entregas:', ln=True)
pdf.set_font('Helvetica', '', 9)
for item in ['Mapas por camada (0-1, normalização robusta)',
             'Ternário K-U-Th', f'PSIIndex ({COMMODITY})',
             f'Painéis por alvo ({" / ".join(ALVOS.keys())})',
             f'Zonas prioritárias (clusters top 5% dentro de {RAIO_KM} km)',
             'Subalvos recomendados (máximos locais espaçados)']:
    pdf.cell(0, 6, f'   • {item}', ln=True)

# --- Páginas com imagens ---
imagens = [
    ('mapa_camadas.png',    'Mapas por Camada — Normalização Robusta 0–1'),
    ('ternario_KUTh.png',   'Ternário Radiométrico K-U-Th (RGB)'),
    ('psiindex.png',        f'PSIIndex ({COMMODITY}) — Favorabilidade Integrada'),
    ('comparacao_alvos.png','Comparação de Alvos'),
]
for alvo_nome in ALVOS:
    imagens.append((f'painel_{alvo_nome}.png', f'Painel Dedicado — {alvo_nome}'))

for img_path, titulo in imagens:
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(0, 8, titulo, ln=True)
        pdf.image(img_path, x=10, w=190)

# --- Página: Tabela de zonas ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 10)
pdf.cell(0, 8, f'Zonas Prioritárias (top 5% por alvo; top 5 por alvo)', ln=True)
pdf.set_font('Courier', '', 7)
header = f"{'Target':<8}{'Zone':>6}{'Priority':>10}{'Peak':>8}{'Mean':>8}{'Area_km2':>10}{'CentLon':>12}{'CentLat':>10}"
pdf.cell(0, 5, header, ln=True)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
for _, row in df_zonas.iterrows():
    linha = (f"{row['Target']:<8}{row['Zone']:>6}{row['PriorityScore']:>10.4f}"
             f"{row['PeakScore']:>8.4f}{row['MeanScore']:>8.4f}"
             f"{row['Area_km2']:>10.4f}{row['CentroidLon']:>12.4f}{row['CentroidLat']:>10.4f}")
    pdf.cell(0, 5, linha, ln=True)

# --- Página: Tabela de subalvos ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 10)
pdf.cell(0, 8, 'Subalvos Recomendados (máximos locais; espaçamento mínimo 5 km)', ln=True)
pdf.set_font('Courier', '', 8)
header2 = f"{'Target':<8}{'Rank':>5}{'Score':>10}{'Lon':>12}{'Lat':>12}"
pdf.cell(0, 6, header2, ln=True)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
for _, row in df_subalvos.iterrows():
    linha2 = f"{row['Target']:<8}{row['Rank']:>5}{row['Score']:>10.5f}{row['Lon']:>12.5f}{row['Lat']:>12.5f}"
    pdf.cell(0, 6, linha2, ln=True)

# --- Página: Conclusão ---
pdf.add_page()
pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 9, f'Conclusão e Interpretação ({COMMODITY} — DADOS SINTÉTICOS)', ln=True)
pdf.set_font('Helvetica', '', 9)
melhor = df_subalvos.sort_values('Score', ascending=False).iloc[0]
pdf.multi_cell(0, 6,
    f'Nesta simulação orientada para {COMMODITY}, o PSIIndex foi construído para privilegiar assinaturas '
    'compatíveis com alteração hidrotermal e controle estrutural: (i) K elevado, (ii) realce de '
    'bordas/gradientes e (iii) bônus em regiões com K alto sem resposta proporcional em MAG/GRAV.\n\n'
    'Interpretação do escore:\n'
    '• PSIIndex é um indicador RELATIVO de favorabilidade, não é teor/reserva.\n'
    '• O ranking é baseado em coerência espacial (zonas contíguas no top 5%), intensidade (PeakScore), '
    'valor médio (MeanScore) e área.\n\n'
    f'Melhor subalvo identificado: {melhor["Target"]} Rank {int(melhor["Rank"])} — '
    f'Score: {melhor["Score"]:.4f} | Lon: {melhor["Lon"]:.5f} | Lat: {melhor["Lat"]:.5f}\n\n'
    'Recomendação (em cenário real): usar as zonas/subalvos como guia para mapeamento e amostragem '
    'de campo, integrando geologia regional e controles estruturais antes de decisões de sondagem.')

# Salvar PDF
NOME_PDF = f'Relatorio_GeoProspecting_{COMMODITY}_pipeline.pdf'
pdf.output(NOME_PDF)
print(f'\n✅ Relatório técnico salvo: {NOME_PDF}')

# Download automático no Colab
try:
    from google.colab import files
    files.download(NOME_PDF)
    files.download('zonas_prioritarias.csv')
    files.download('subalvos_recomendados.csv')
    print('✅ Download iniciado automaticamente!')
except ImportError:
    print('ℹ️  Fora do Colab — arquivos salvos no diretório atual.')

## 12. Resumo Final e Próximos Passos

---

### O que este pipeline entrega

| Produto | Arquivo |
|---|---|
| Mapas por camada (normalização robusta) | `mapa_camadas.png` |
| Ternário radiométrico K-U-Th | `ternario_KUTh.png` |
| Mapa do PSIIndex | `psiindex.png` |
| Painéis por alvo | `painel_T1/T2/T3.png` |
| Zonas prioritárias (CSV) | `zonas_prioritarias.csv` |
| Subalvos recomendados (CSV) | `subalvos_recomendados.csv` |
| Relatório técnico completo | `Relatorio_GeoProspecting_OURO_pipeline.pdf` |

---

### Limitações importantes
- O PSIIndex é um **indicador relativo** de favorabilidade — não é teor, reserva ou laudo geológico
- Os resultados dependem diretamente da **qualidade e resolução dos dados** fornecidos
- Deve ser usado como **ferramenta de apoio à decisão**, integrado com mapeamento geológico, geoquímica e trabalhos de campo

### Próximos passos recomendados
1. **Substituir dados sintéticos** por dados geofísicos reais (GeoTIFF ou CSV)
2. **Ajustar os pesos** do PSIIndex conforme o modelo geológico regional da área
3. **Validar** as zonas prioritárias com mapeamento geológico de superfície
4. Executar **amostragem geoquímica** nos subalvos top-ranked
5. Planejar **linhas de sondagem** com base nas zonas confirmadas

---
> ⚠️ **IMPORTANTE:** Este notebook usa dados **SINTÉTICOS**. Todos os resultados são para fins de demonstração metodológica.